# Changeover sales impact
For each GTIN changeover (old → new GTIN in `silver.product_launches`), estimate how weekly unit sales changed **because of** the change, separate from seasonality, trend and promotions.

* **Stitched series:** old + new GTIN units per product and store (`silver.weekly_sales`)
* **Windows:** 26 weeks before the cutover; weeks 1-3 = transition (reported as the transition dip); weeks 4-15 = after
* **Model:** OLS of log(units + 1) with product×store and week fixed effects, `on_promo`, treated×transition and treated×post; effect = exp(β) − 1
* **Controls:** same-category products with no GTIN change within ±26 weeks (from the data)
* **95% range:** placebo runs (each control treated as if it changed on the same date) give the spread of chance effects
* **Controls fit:** no significant pre-period trend between product and controls (p ≥ 0.05)
* **Status:** measured, pending (cutover in the future) or insufficient_data

Writes `gold.changeover_impact` (one row per changeover) and `gold.changeover_weekly` (actual vs expected weekly units). Sales are synthetic demo data; `silver.synthetic_changeover_truth` is used only for the accuracy check.

In [0]:
%pip install -q statsmodels
dbutils.library.restartPython()

In [0]:
import warnings
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

dbutils.widgets.text("catalog", "patricia_aws_serverless_catalog")
CATALOG = dbutils.widgets.get("catalog")
PRE_WEEKS, TRANSITION_WEEKS, POST_WEEKS = 26, 3, 12

co = spark.sql(f"""
  SELECT client_id, article_description, identifier_old AS gtin_old, identifier_new AS gtin_new,
         CAST(DATE_TRUNC('week', launch_date) AS DATE) AS launch_week, source_file
  FROM {CATALOG}.silver.product_launches
  WHERE identifier_type = 'GTIN' AND identifier_old IS NOT NULL AND identifier_new IS NOT NULL
    AND trim(identifier_old) <> '' AND identifier_old <> identifier_new""").toPandas()
co["launch_week"] = pd.to_datetime(co["launch_week"])
assert not co.duplicated("gtin_old").any() and not co.duplicated("gtin_new").any(), "a GTIN appears in two changeovers"

sales = spark.table(f"{CATALOG}.silver.weekly_sales").toPandas()
sales["week"] = pd.to_datetime(sales["week"])
LAST_WEEK = sales.week.max()

# Stitch: the new GTIN's sales count toward the product identified by its old GTIN
sales["product_key"] = sales.gtin.map(dict(zip(co.gtin_new, co.gtin_old))).fillna(sales.gtin)
panel = (sales.groupby(["product_key", "store_id", "week"], as_index=False)
              .agg(units=("units", "sum"), revenue=("revenue_eur", "sum"), on_promo=("on_promo", "max"), n_gtins=("gtin", "nunique"),
                   category=("category", "first"), client_id=("client_id", "first"), product=("article_description", "first")))
products = panel.groupby("product_key").agg(category=("category", "first"), first_week=("week", "min")).reset_index()
co = co.merge(products, left_on="gtin_old", right_on="product_key", how="left")

def status(r):
    if pd.isna(r.first_week): return "insufficient_data"
    if r.launch_week > LAST_WEEK: return "pending"
    pre_ok = r.launch_week - pd.Timedelta(weeks=PRE_WEEKS) >= r.first_week
    post_ok = r.launch_week + pd.Timedelta(weeks=TRANSITION_WEEKS + POST_WEEKS - 1) <= LAST_WEEK
    return "measured" if pre_ok and post_ok else "insufficient_data"
co["status"] = co.apply(status, axis=1)

changed = co.set_index("gtin_old")["launch_week"]
def controls_for(r):
    lo, hi = r.launch_week - pd.Timedelta(weeks=PRE_WEEKS), r.launch_week + pd.Timedelta(weeks=TRANSITION_WEEKS + POST_WEEKS)
    cands = products[(products.category == r.category) & (products.product_key != r.gtin_old)].product_key
    return [k for k in cands if not (k in changed.index and lo <= changed[k] <= hi)]
co["controls"] = co.apply(controls_for, axis=1)
co.loc[(co.status == "measured") & (co.controls.str.len() == 0), "status"] = "insufficient_data"

assert int(sales.units.sum()) == int(panel.units.sum()), "stitching changed the unit total"
print("last sales week:", LAST_WEEK.date(), "| status:", co.status.value_counts().to_dict())
display(co[["client_id", "article_description", "launch_week", "status", "category"]].assign(n_controls=co.controls.str.len()))

In [0]:
def window(treated, controls, c):
    lo, hi = c - pd.Timedelta(weeks=PRE_WEEKS), c + pd.Timedelta(weeks=TRANSITION_WEEKS + POST_WEEKS)
    d = panel[panel.product_key.isin([treated] + list(controls)) & (panel.week >= lo) & (panel.week < hi)].copy()
    k = (d.week - c).dt.days // 7
    d["treated"] = (d.product_key == treated).astype(int)
    d["tr_transition"] = d.treated * k.between(0, TRANSITION_WEEKS - 1).astype(int)
    d["tr_post"] = d.treated * (k >= TRANSITION_WEEKS).astype(int)
    d["ps"], d["wk"], d["promo"], d["t"] = d.product_key + "|" + d.store_id, d.week.dt.strftime("%Y-%m-%d"), d.on_promo.astype(int), k
    d["ly"], d["lrev"] = np.log1p(d.units), np.log1p(d.revenue)
    return d

def did(treated, controls, c, y="ly"):
    d = window(treated, controls, c)
    return smf.ols(f"{y} ~ tr_post + tr_transition + promo + C(ps) + C(wk)", data=d).fit(), d

def pretrend_p(treated, controls, c):
    d = window(treated, controls, c)
    d = d[d.t < 0].assign(tr_t=lambda x: x.treated * x.t)
    m = smf.ols("ly ~ tr_t + promo + C(ps) + C(wk)", data=d).fit()
    return float(m.pvalues["tr_t"])

measured = co[co.status == "measured"].reset_index(drop=True)
est, placebos, models = [], [], {}
for r in measured.itertuples():
    m, d = did(r.gtin_old, r.controls, r.launch_week)
    mr, _ = did(r.gtin_old, r.controls, r.launch_week, y="lrev")
    models[r.gtin_old] = (m, d)
    est.append(dict(gtin_old=r.gtin_old, beta=m.params["tr_post"], dip=m.params["tr_transition"], beta_rev=mr.params["tr_post"],
                    pretrend_p=pretrend_p(r.gtin_old, r.controls, r.launch_week)))
    for k in r.controls:  # placebo: a control "changes" on the same date; the real changed product is left out
        others = [x for x in r.controls if x != k]
        if others:
            placebos.append(did(k, others, r.launch_week)[0].params["tr_post"])
est = pd.DataFrame(est)
q_lo, q_hi = np.percentile(placebos, [2.5, 97.5])
est = est.assign(effect_pct=np.expm1(est.beta) * 100, ci_low=np.expm1(est.beta - q_hi) * 100, ci_high=np.expm1(est.beta - q_lo) * 100,
                 transition_dip_pct=np.expm1(est.dip) * 100, revenue_effect_pct=np.expm1(est.beta_rev) * 100, controls_fit=est.pretrend_p >= 0.05)
print(f"{len(placebos)} placebo runs: 95% band of chance effects {np.expm1(q_lo) * 100:+.1f}% .. {np.expm1(q_hi) * 100:+.1f}%")
display(measured[["article_description"]].join(est.drop(columns=["gtin_old", "beta", "dip", "beta_rev"])).round(2))

In [0]:
weekly = []
for r in measured.itertuples():
    m, d = models[r.gtin_old]
    tr = d[d.treated == 1].copy()
    tr["expected"] = np.expm1(m.predict(tr.assign(tr_post=0, tr_transition=0)))  # same product without the change
    w = tr.groupby("week", as_index=False).agg(actual_units=("units", "sum"), expected_units=("expected", "sum"))
    k = (w.week - r.launch_week).dt.days // 7
    w["phase"] = np.select([k < 0, k < TRANSITION_WEEKS], ["before", "transition"], "after")
    weekly.append(w.assign(gtin_old=r.gtin_old))
for r in co[co.status != "measured"].itertuples():  # pending: recent actuals only
    w = panel[(panel.product_key == r.gtin_old) & (panel.week > LAST_WEEK - pd.Timedelta(weeks=PRE_WEEKS))].groupby("week", as_index=False).agg(actual_units=("units", "sum"))
    weekly.append(w.assign(expected_units=np.nan, phase="before", gtin_old=r.gtin_old))
weekly = pd.concat(weekly, ignore_index=True)

rates = weekly.groupby(["gtin_old", "phase"]).actual_units.mean().unstack()
baseline = weekly[weekly.phase == "before"].sort_values("week").groupby("gtin_old").tail(12).groupby("gtin_old").actual_units.mean()

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for ax, r in zip(axes, measured.head(2).itertuples()):
    w = weekly[weekly.gtin_old == r.gtin_old]
    ax.plot(w.week, w.actual_units, marker=".", color="#1b3139", label="actual")
    ax.plot(w.week, w.expected_units, ls="--", color="#ff3621", label="expected without the change")
    ax.axvspan(r.launch_week, r.launch_week + pd.Timedelta(weeks=TRANSITION_WEEKS), color="grey", alpha=.15, label="transition")
    ax.set_title(r.article_description, fontsize=10); ax.legend(fontsize=8)
fig.tight_layout(); display(fig)

In [0]:
truth = spark.table(f"{CATALOG}.silver.synthetic_changeover_truth").toPandas()[["gtin_old", "planted_effect_pct"]]
v = measured[["article_description", "gtin_old"]].join(est[["effect_pct", "ci_low", "ci_high", "controls_fit"]]).merge(truth, on="gtin_old", how="left")
v = v.assign(error_pts=v.effect_pct - v.planted_effect_pct,
             planted_in_range=(v.planted_effect_pct >= v.ci_low) & (v.planted_effect_pct <= v.ci_high))
display(v.drop(columns="gtin_old").round(2))
print(f"mean abs error {v.error_pts.abs().mean():.2f} pts | planted effect inside the 95% range: {int(v.planted_in_range.sum())}/{len(v)}")

In [0]:
now = pd.Timestamp.utcnow().tz_localize(None)
impact = co.merge(est.drop(columns=["beta", "dip", "beta_rev"]), on="gtin_old", how="left")
impact = impact.assign(
    launch_date=impact.launch_week.dt.date,
    units_per_week_pre=impact.gtin_old.map(rates["before"]).where(impact.status == "measured", impact.gtin_old.map(baseline)),
    units_per_week_post=impact.gtin_old.map(rates.get("after")) if "after" in rates else np.nan,
    n_controls=impact.controls.str.len(),
    significant=(impact.ci_low > 0) | (impact.ci_high < 0),
    method=f"DiD OLS log1p(units), product-store + week FE, on_promo; {PRE_WEEKS}w before, weeks {TRANSITION_WEEKS + 1}-{TRANSITION_WEEKS + POST_WEEKS} after; placebo 95% range",
    is_synthetic_data=True, evaluated_at=now)
impact.loc[impact.status != "measured", ["units_per_week_post", "significant"]] = [np.nan, None]
impact = impact[["client_id", "article_description", "category", "gtin_old", "gtin_new", "launch_date", "status", "effect_pct", "ci_low", "ci_high",
                 "significant", "transition_dip_pct", "revenue_effect_pct", "units_per_week_pre", "units_per_week_post", "n_controls",
                 "controls_fit", "pretrend_p", "method", "is_synthetic_data", "evaluated_at"]]
meas = impact[impact.status == "measured"]
assert meas[["effect_pct", "ci_low", "ci_high"]].notna().all().all(), "measured changeover without an effect"
spark.createDataFrame(impact).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.changeover_impact")
spark.sql(f"COMMENT ON TABLE {CATALOG}.gold.changeover_impact IS 'Sales impact per GTIN changeover (effect % with 95% placebo range). Written by src/analysis/changeover_impact; based on synthetic sales.'")

wk = weekly.merge(co[["gtin_old", "gtin_new", "client_id", "article_description"]], on="gtin_old").assign(week=lambda x: x.week.dt.date, evaluated_at=now)
spark.createDataFrame(wk[["client_id", "article_description", "gtin_old", "gtin_new", "week", "phase", "actual_units", "expected_units", "evaluated_at"]]) \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.changeover_weekly")
spark.sql(f"COMMENT ON TABLE {CATALOG}.gold.changeover_weekly IS 'Actual vs expected (no change) weekly units per changeover, for charts. Written by src/analysis/changeover_impact.'")
display(spark.table(f"{CATALOG}.gold.changeover_impact").orderBy("status", "launch_date"))